# 🎬 Supernan AI Intern – Hindi Dubbing Pipeline
**Google Colab Notebook (Free T4 GPU)**

> **Runtime**: Set Runtime → **T4 GPU** before running.
> **Source language**: Kannada (auto-detected). Clip window: **0:45 – 1:00** (confirmed speech).

## Cell 1 – Install System Packages

In [ ]:
!apt-get install -y -q ffmpeg rubberband-cli
!ffmpeg -version | head -1
print('✓ System packages ready')

## Cell 2 – Install Python Dependencies

**Fix**: Coqui `TTS` PyPI package requires Python <3.12 (Colab now uses 3.12).
We install from the community-maintained GitHub fork which supports 3.12.

In [ ]:
# Core utilities
!pip install -q ffmpeg-python pydub colorlog soundfile pyrubberband numpy
print('✓ Core utilities installed')

# Whisper (ASR)
!pip install -q openai-whisper
print('✓ Whisper installed')

# Translation
!pip install -q deep-translator
print('✓ Translator installed')

# Coqui XTTS v2 — install from GitHub (supports Python 3.12)
!pip install -q git+https://github.com/idiap/coqui-ai-tts
print('✓ Coqui XTTS v2 installed')

# Face restoration
!pip install -q facexlib basicsr gfpgan
print('✓ GFPGAN installed')

# HuggingFace / IndicTrans2 runtime
!pip install -q huggingface_hub transformers sentencepiece sacremoses
print('✓ HuggingFace stack installed')

# OpenCV for frame extraction
!pip install -q opencv-python-headless
print('✓ OpenCV installed')

## Cell 3 – Install IndicTrans2

**Fix**: The correct install target is the `huggingface_interface` subdirectory.

In [ ]:
!pip install -q \
    git+https://github.com/AI4Bharat/IndicTrans2.git#subdirectory=huggingface_interface
print('✓ IndicTrans2 installed')

## Cell 4 – Clone VideoReTalking

In [ ]:
import os

if not os.path.exists('video-retalking'):
    !git clone https://github.com/vinthony/video-retalking.git

if not os.path.exists('video-retalking/checkpoints'):
    os.makedirs('video-retalking/checkpoints', exist_ok=True)
    from huggingface_hub import snapshot_download
    snapshot_download(
        'vinthony/video-retalking',
        local_dir='video-retalking/checkpoints',
        ignore_patterns=['*.git*'],
    )

# Install VideoReTalking deps (ignore build errors for optional packages)
!pip install -q -r video-retalking/requirements.txt 2>&1 | grep -v 'ERROR\|error' || true
print('✓ VideoReTalking ready')

## Cell 5 – Clone Pipeline Repo

In [ ]:
REPO_URL = 'https://github.com/sudip-kumar-prasad/supernan-hindi-dubbing-pipeline.git'

if not os.path.exists('supernan-hindi-dubbing-pipeline'):
    !git clone {REPO_URL}

import sys
sys.path.insert(0, '/content/supernan-hindi-dubbing-pipeline')
print('✓ Pipeline repo ready')

## Cell 6 – Download Source Video

In [ ]:
!pip install -q gdown

# Option A: Google Drive (default)
FILE_ID = '1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW'
!gdown {FILE_ID} -O /content/input.mp4

# Option B: if gdown fails, uncomment and manually upload:
# from google.colab import files; files.upload()

import os
assert os.path.exists('/content/input.mp4'), 'input.mp4 not found'
print(f'✓ Video ready: {os.path.getsize("/content/input.mp4") / 1e6:.1f} MB')

## Cell 7 – Run the Pipeline

In [ ]:
os.chdir('/content/supernan-hindi-dubbing-pipeline')

# 0:45–1:00 = confirmed Kannada speech window. large-v3 for best Kannada ASR.
!python dub_video.py \
    --input /content/input.mp4 \
    --output /content/output.mp4 \
    --start 45 \
    --end 60 \
    --model large-v3 \
    --verbose

## Cell 8 – Preview & Download Output

In [ ]:
from IPython.display import Video, display
import os

output_path = '/content/output.mp4'
assert os.path.exists(output_path), 'output.mp4 not found – check pipeline logs in Cell 7'
print(f'Output: {os.path.getsize(output_path) / 1e6:.1f} MB')
display(Video(output_path, embed=True, width=640))

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

---
## 💡 Tips

- Source video is **Kannada** — `large-v3` handles it well, `base` will miss segments.
- If a cell crashes, just re-run it — pipeline resumes from last checkpoint automatically.
- Add `--batch` flag for videos longer than 5 minutes (silence-based audio chunking).
- If VideoReTalking OOMs, the pipeline auto-falls back to Wav2Lip.
- If lip-sync/enhance are too slow on free tier, add `--skip-lipsync --skip-enhance` for a quick audio-dubbed test video.